# Classes, tree-sitter module: crm-prototype (javascript)
Module: `core-module/classes.py`. Each rule must show the rows it fixes; a rule that fixes nothing here is still listed with 0.

In [1]:
import pandas as pd, class_lab as L
pd.set_option('display.max_colwidth', 100); pd.set_option('display.width', 250)
APP, LANG = 'crm-prototype', 'javascript'
REPO = L.ROOT / 'apps' / APP
DB = L.ROOT / 'results' / APP / 'runs/iter-003_2026-11-18_0743/crg/graph.db'
files = L.repo_files(REPO, LANG)
print(LANG, 'files:', len(files))

javascript files: 89


## 1. Reference: TypeScript compiler (`js-ref/classes.mjs`)
Named classes are expected. Anonymous classes have no name and are not expected.

In [2]:
ref = L.reference(REPO, LANG, files)
print('parse errors:', ref.error.notna().sum())
ref = ref[ref.error.isna()]
print(ref.kind.value_counts().to_string())
named = ref[ref.kind != 'anonymous'].reset_index(drop=True)
print('expected named classes:', len(named))

parse errors: 0
kind
class    6
expected named classes: 6


## 2. Module with no rules
The plain tree-sitter result, before any rule.

In [3]:
m0, m0_miss, m0_extra = L.score(named, L.module(REPO, LANG, files, rules=frozenset()))
pd.Series(m0).to_frame('module, no rules')

,"module, no rules"
expected,6.0
found,6.0
found %,100.0
same end line,6.0
missed,0.0
extra,0.0


In [4]:
print('missed'); display(m0_miss.assign(source=[L.source_line(REPO, f, l) for f, l in zip(m0_miss.file, m0_miss.line)]))
print('extra'); display(m0_extra.assign(source=[L.source_line(REPO, f, l) for f, l in zip(m0_extra.file, m0_extra.line_start)]))

missed


,file,name,line,end,kind,source


extra


,file,name,line_start,line_end,source


## 3. Proof per rule
For each rule: the module is run with every rule **except** that one, then with all rules. The rows that change are the rows the rule fixes.

In [5]:
rules, fixed = L.rule_proof(REPO, LANG, files, named)
rules

,rule,what,found without rule,found with all rules,rows fixed (missed -> found),wrong rows removed
0,1,java: annotation types (@interface) are classes,6,6,0,0
1,2,js: class expression named by the variable/property it is assigned to,6,6,0,0
2,3,python: decorated class starts at its first decorator,6,6,0,0


In [6]:
fixed

,rule,fix,file,name,line,source


## 4. Module with all rules

In [7]:
m, m_miss, m_extra = L.score(named, L.module(REPO, LANG, files))
pd.DataFrame({'no rules': m0, 'all rules': m})

,no rules,all rules
expected,6.0,6.0
found,6.0,6.0
found %,100.0,100.0
same end line,6.0,6.0
missed,0.0,0.0
extra,0.0,0.0


In [8]:
print('still missed'); display(m_miss)
print('still extra'); display(m_extra)

still missed


,file,name,line,end,kind


still extra


,file,name,line_start,line_end


## 5. False positives
**False positive** = a module class whose name is not a class in that file in the reference (any line).

In [9]:
m_all = L.module(REPO, LANG, files)
m_fp = L.false_positives(named, m_all, REPO)
print('false positives:', len(m_fp))
m_fp

false positives: 0


,file,name,line_start,line_end,source
